# **Kode asli pong game dengan hand gesture**

In [1]:
import cv2
import cvzone
from cvzone.HandTrackingModule import HandDetector
import numpy as np

cap = cv2.VideoCapture(0)
cap.set(3, 1280)
cap.set(4, 720)

# Importing all images
imgBackground = cv2.imread("Resources/Background.png")
imgGameOver = cv2.imread("Resources/gameOver.png")
imgBall = cv2.imread("Resources/Ball.png", cv2.IMREAD_UNCHANGED)
imgBat1 = cv2.imread("Resources/bat1.png", cv2.IMREAD_UNCHANGED)
imgBat2 = cv2.imread("Resources/bat2.png", cv2.IMREAD_UNCHANGED)

# Hand Detector
detector = HandDetector(detectionCon=0.8, maxHands=2)

# Variables
ballPos = [100, 100]
speedX = 15
speedY = 15
gameOver = False
score = [0, 0]

while True:
    _, img = cap.read()
    img = cv2.flip(img, 1)
    imgRaw = img.copy()

    # Find the hand and its landmarks
    hands, img = detector.findHands(img, flipType=False)  # with draw

    # Overlaying the background image
    img = cv2.addWeighted(img, 0.2, imgBackground, 0.8, 0)

    # Check for hands
    if hands:
        for hand in hands:
            x, y, w, h = hand['bbox']
            h1, w1, _ = imgBat1.shape
            y1 = y - h1 // 2
            y1 = np.clip(y1, 20, 415)

            if hand['type'] == "Left":
                img = cvzone.overlayPNG(img, imgBat1, (59, y1))
                if 59 < ballPos[0] < 59 + w1 and y1 < ballPos[1] < y1 + h1:
                    speedX = -speedX
                    ballPos[0] += 30
                    score[0] += 1

            if hand['type'] == "Right":
                img = cvzone.overlayPNG(img, imgBat2, (1195, y1))
                if 1195 - 50 < ballPos[0] < 1195 and y1 < ballPos[1] < y1 + h1:
                    speedX = -speedX
                    ballPos[0] -= 30
                    score[1] += 1

    # Game Over
    if ballPos[0] < 40 or ballPos[0] > 1200:
        gameOver = True

    if gameOver:
        img = imgGameOver
        cv2.putText(img, str(score[1] + score[0]).zfill(2), (585, 360), cv2.FONT_HERSHEY_COMPLEX,
                    2.5, (200, 0, 200), 5)

    # If game not over move the ball
    else:

        # Move the Ball
        if ballPos[1] >= 500 or ballPos[1] <= 10:
            speedY = -speedY

        ballPos[0] += speedX
        ballPos[1] += speedY

        # Draw the ball
        img = cvzone.overlayPNG(img, imgBall, ballPos)

        cv2.putText(img, str(score[0]), (300, 650), cv2.FONT_HERSHEY_COMPLEX, 3, (255, 255, 255), 5)
        cv2.putText(img, str(score[1]), (900, 650), cv2.FONT_HERSHEY_COMPLEX, 3, (255, 255, 255), 5)

    img[580:700, 20:233] = cv2.resize(imgRaw, (213, 120))

    cv2.imshow("Image", img)
    key = cv2.waitKey(1)
    if key == ord('r'):
        ballPos = [100, 100]
        speedX = 15
        speedY = 15
        gameOver = False
        score = [0, 0]
        imgGameOver = cv2.imread("Resources/gameOver.png")

C:\Users\ADVAN\anaconda3\envs\Kuliah-Ai\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


KeyboardInterrupt: 

# **Kode pong game dengan hand gesture**
Penambahan multiplier kecepatan bola, aturan game over setelah mencetak 5 skor

In [2]:
import cv2
import cvzone
from cvzone.HandTrackingModule import HandDetector
import numpy as np

# Webcam
cap = cv2.VideoCapture(0)
cap.set(3, 1280)
cap.set(4, 720)

# Images
imgBackground = cv2.imread("Resources/Background.png")
imgGameOver = cv2.imread("Resources/gameOver2.png")
imgBall = cv2.imread("Resources/Ball.png", cv2.IMREAD_UNCHANGED)
imgBat1 = cv2.imread("Resources/bat1.png", cv2.IMREAD_UNCHANGED)
imgBat2 = cv2.imread("Resources/bat2.png", cv2.IMREAD_UNCHANGED)

# Hand Detector
detector = HandDetector(detectionCon=0.8, maxHands=2)

# Game Variables
ballPos = [640, 360]
speedX = 10
speedY = 10

score = [0, 0]
maxScore = 5

gameOver = False
winner = ""

# Ball Settings
speedMultiplier = 1.1
maxSpeed = 40

while True:

    success, img = cap.read()

    if not success:
        break

    img = cv2.flip(img, 1)
    imgRaw = img.copy()

    # Hand Detection
    hands, img = detector.findHands(img, flipType=False)

    # Background
    img = cv2.addWeighted(img, 0.2, imgBackground, 0.8, 0)

    # =========================
    # GAME LOOP
    # =========================

    if not gameOver:

        # -------------------------
        # PLAYER PADDLES
        # -------------------------

        if hands:
            for hand in hands:

                x, y, w, h = hand['bbox']

                h1, w1, _ = imgBat1.shape

                y1 = y - h1 // 2
                y1 = np.clip(y1, 20, 415)

                # LEFT PLAYER
                if hand['type'] == "Left":

                    img = cvzone.overlayPNG(img, imgBat1, (59, y1))

                    if (
                        59 < ballPos[0] < 59 + w1 and
                        y1 < ballPos[1] < y1 + h1
                    ):

                        speedX = -speedX

                        # Speed Increase
                        speedX *= speedMultiplier
                        speedY *= speedMultiplier

                        # Clamp Speed
                        speedX = np.clip(speedX, -maxSpeed, maxSpeed)
                        speedY = np.clip(speedY, -maxSpeed, maxSpeed)

                        ballPos[0] += 30

                # RIGHT PLAYER
                if hand['type'] == "Right":

                    img = cvzone.overlayPNG(img, imgBat2, (1195, y1))

                    if (
                        1195 - 50 < ballPos[0] < 1195 and
                        y1 < ballPos[1] < y1 + h1
                    ):

                        speedX = -speedX

                        # Speed Increase
                        speedX *= speedMultiplier
                        speedY *= speedMultiplier

                        # Clamp Speed
                        speedX = np.clip(speedX, -maxSpeed, maxSpeed)
                        speedY = np.clip(speedY, -maxSpeed, maxSpeed)

                        ballPos[0] -= 30

        # -------------------------
        # BALL MOVEMENT
        # -------------------------

        if ballPos[1] >= 500 or ballPos[1] <= 10:
            speedY = -speedY

        ballPos[0] += int(speedX)
        ballPos[1] += int(speedY)

        # -------------------------
        # GOAL DETECTION
        # -------------------------

        # Goal kiri -> poin kanan
        if ballPos[0] <= 0:

            score[1] += 1

            ballPos = [640, 360]

            speedX = 15
            speedY = np.random.choice([-15, 15])

        # Goal kanan -> poin kiri
        if ballPos[0] >= 1280:

            score[0] += 1

            ballPos = [640, 360]

            speedX = -15
            speedY = np.random.choice([-15, 15])

        # -------------------------
        # WIN CONDITION
        # -------------------------

        if score[0] >= maxScore:
            gameOver = True
            winner = "LEFT WINS"

        if score[1] >= maxScore:
            gameOver = True
            winner = "RIGHT WINS"

        # -------------------------
        # DRAW BALL
        # -------------------------

        if 0 <= ballPos[0] <= 1230 and 0 <= ballPos[1] <= 670:
            img = cvzone.overlayPNG(img, imgBall, ballPos)

        # -------------------------
        # DRAW SCORE
        # -------------------------

        cv2.putText(
            img,
            str(score[0]),
            (300, 650),
            cv2.FONT_HERSHEY_COMPLEX,
            3,
            (255, 255, 255),
            5
        )

        cv2.putText(
            img,
            str(score[1]),
            (900, 650),
            cv2.FONT_HERSHEY_COMPLEX,
            3,
            (255, 255, 255),
            5
        )

    # =========================
    # GAME OVER SCREEN
    # =========================

    else:
        img = imgGameOver.copy()
    
        # Winner Text
        cv2.putText(
            img,
            winner,
            (530, 260),
            cv2.FONT_HERSHEY_DUPLEX,
            1.3,
            (255, 255, 255),
            3
        )
    
        # Final Score
        finalScore = f"{score[0]} : {score[1]}"
    
        cv2.putText(
            img,
            finalScore,
            (560, 355),
            cv2.FONT_HERSHEY_COMPLEX,
            2.5,
            (200, 0, 200),
            5
        )

    # Restart Instruction
    cv2.putText(
        img,
        "Press R to restart",
        (500, 520),
        cv2.FONT_HERSHEY_COMPLEX,
        0.9,
        (255, 255, 255),
        2
    )

    # Webcam Preview
    img[580:700, 20:233] = cv2.resize(imgRaw, (213, 120))

    cv2.imshow("Image", img)

    key = cv2.waitKey(1)

    # Restart Game
    if key == ord('r'):

        ballPos = [640, 360]

        speedX = 15
        speedY = 15

        score = [0, 0]

        gameOver = False
        winner = ""

    # Quit
    if key == 27:
        break

cap.release()
cv2.destroyAllWindows()